# Credit Card Fraud Detection

Credit cards are frequently used due to their convenience in facilitating transactions, offering a quick and secure way for individuals to make purchases, both online and in-person. The importance of fraud detection arises from the potential risks associated with unauthorized transactions, as it helps safeguard users from financial losses and ensures the integrity of the overall payment system. To address this, a system is needed to track transaction patterns and automatically abort suspicious transactions. In this assigment, you train machine learning models using past data to effectively classify transactions as normal or abnormal.

Import basic libraries.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

Load the file dataset to a pandas DataFrame

In [ ]:
# Use local copy when available; otherwise download from GitHub

if not os.path.exists('creditcard.csv.gz'):
    url = "https://raw.githubusercontent.com/vanraak/data2026/main/creditcard.csv.gz"
    df = pd.read_csv(url, compression="gzip")
else:
    df = pd.read_csv("creditcard.csv.gz", compression="gzip")

## Data Understanding

Display top rows of the DataFrame

In [ ]:
df.head()

Display the number of fraud and non-fraud cases in the dataset, using .value_counts() following the DataFrame column

In [ ]:
df["class"].value_counts()

Compute the percentage of observations that pertain to fraud.

In [ ]:
df["class"].value_counts()

Display the descriptive statistics for the dataset.

In [ ]:
df.describe().T

Compare fraud and non-fraud cases in terms of amounts of the transactions. Use the .groupby() command to group by `class` (fraud vs. non-fraud). Only show the descriptive statistics for the 'Amount' column.

In [ ]:
df.groupby("class")["amount"].mean()

Examine the distribution of `amount`.

In [ ]:
sns.histplot(df, x="amount", bins=30)

## Data Preparation

Apply a logarithmic transformation to the amount variable.

In [ ]:
df["log_amount"]=np.log1p(df["amount"])

Examine the distribution after applying the log transformation.

In [ ]:
sns.histplot(df, x="log_amount")

Split your DataFrame into two parts:

- *X* which contains all the features (explanatory variables - excluding amount)<br>
- *y* which is your outcome variable (fraud).

In [ ]:
X = df.drop(columns=["class", "amount"])
y = df["class"]

Split your X and y datasets into training and test datasets:

  Use 80% of your sample for training and 20% of your sample for testing.

  You can use the train_test_split function from sklearn.model_selection to do this.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.10, stratify=y)

As you noticed, the scales of your variables vary considerably. Normalize the columns in your X vectors. Normalization means that you change the range of each variable from 0 to 1.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().set_output(transform="pandas")
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Modeling: KNN

15) K-Nearest Neigbors:
- Import the k-Nearest neighbors classifier algorithm from scikit-learn.
- Import *ConfusionMatrixDisplay*, *precision_score* and *recall_score* from sklearn.metrics
- Create an instance of the model and fit it to the training data
- Compute: 1) the accuracy of the model on the test data, 2) the precision score for the test data, and 3) the recall score for the test data.
- Display a confusion matrix
- Decide on the appropriate 'k' for the the algorithm.

Utilize print statements with f-strings to neatly display the scores.

Import the k-Nearest neighbors classifier algorithm from scikit-learn.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

Create an instance of the model.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=3)

Fit the model to the training data

In [ ]:
knn.fit(X_train, y_train)

Generate predictions for the test data

In [ ]:
y_pred = knn.predict(X_test)

## Evaluation: KNN

Import `ConfusionMatrixDisplay` from `sklearn.metrics` and generate a confusion matrix.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)

- Import `accuracy_score`, `precision_score` and `recall_score` from `sklearn.metrics`.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score


print(f"Accuracy score:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision score: {precision_score(y_test,y_pred):.4f}")
print(f"Recall score:    {recall_score(y_test,y_pred):.4f}")

### Try it yourself

- Choose a different $k$ and re-run the analyses.

In [ ]:
k = 9 # Change this number


knn = KNeighborsClassifier(n_neighbors=k)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
print(f"Accuracy score:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision score: {precision_score(y_test,y_pred):.4f}")
print(f"Recall score:    {recall_score(y_test,y_pred):.4f}")

- Use a custom classification threshold. How does that change the results?

In [ ]:
threshold = 0.3 # Change this number
k = 3 # Change this number


knn = KNeighborsClassifier(n_neighbors=k)
knn.fit(X_train, y_train)
y_pred = knn.predict_proba(X_test)[:, 1] >= threshold

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
print(f"Accuracy score:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision score: {precision_score(y_test,y_pred):.4f}")
print(f"Recall score:    {recall_score(y_test,y_pred):.4f}")


## Decision Tree

Import the decision tree classifier algorithm from scikit-learn.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

Create an instance of the model and fit it to the training data

In [ ]:
tree = DecisionTreeClassifier()
tree.fit(X_train, y_train)

Visualize the decision tree.

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(12, 8))

plot_tree(
    tree,
    feature_names=X_train.columns,
    class_names=["0", "1"],
    filled=True
)

plt.show()

Decide on the maximum number of layers and refit the model.

In [ ]:
tree = DecisionTreeClassifier(max_depth=4)
tree.fit(X_train, y_train)

Generate the predictions

In [ ]:
y_pred = tree.predict(X_test)

Display a confusion matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)

Compute the accuracy score, precision score and recall score.

In [ ]:
print(f"Accuracy score:  {accuracy_score(y_test,y_pred):.4f}")
print(f"Precision score: {precision_score(y_test,y_pred):.4f}")
print(f"Recall score:    {recall_score(y_test,y_pred):.4f}")

### Try it yourself

- Instead of controlling tree complexity only with `max_depth`, experiment with other pruning strategies.

> > ◦ Use `max_leaf_nodes` to limit the number of decision regions.

In [ ]:
max_leaves = 20 #change this number
threshold = 0.5 #change this number

tree = DecisionTreeClassifier(max_leaf_nodes=10)
tree.fit(X_train, y_train)
y_pred = tree.predict_proba(X_test)[:,1]>=threshold

In [ ]:
plt.figure(figsize=(12, 8))

plot_tree(
    tree,
    feature_names=X_train.columns,
    class_names=["0", "1"],
    filled=True
)

plt.show()

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
print(f"Accuracy score:  {accuracy_score(y_test,y_pred):.4f}")
print(f"Precision score: {precision_score(y_test,y_pred):.4f}")
print(f"Recall score:    {recall_score(y_test,y_pred):.4f}")

> > ◦ Try cost-complexity pruning using `ccp_alpha` and compare its effect with pre-pruning approaches.

In [ ]:
ccp_alpha =0.0001 #change this number
threshold = 0.5 #change this number

tree = DecisionTreeClassifier(ccp_alpha=ccp_alpha)
tree.fit(X_train, y_train)
y_pred = tree.predict_proba(X_test)[:,1]>=threshold

In [ ]:
plt.figure(figsize=(12, 8))

plot_tree(
    tree,
    feature_names=X_train.columns,
    class_names=["0", "1"],
    filled=True
)

plt.show()

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
print(f"Accuracy score:  {accuracy_score(y_test,y_pred):.4f}")
print(f"Precision score: {precision_score(y_test,y_pred):.4f}")
print(f"Recall score:    {recall_score(y_test,y_pred):.4f}")

> > ◦ In imbalanced classification problems, the majority class can dominate the model. Experiment with *class weighting* to give more importance to the minority class.

In [ ]:
tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced")
tree.fit(X_train, y_train)
y_pred = tree.predict_proba(X_test)[:,1]>=threshold

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
print(f"Accuracy score:  {accuracy_score(y_test,y_pred):.4f}")
print(f"Precision score: {precision_score(y_test,y_pred):.4f}")
print(f"Recall score:    {recall_score(y_test,y_pred):.4f}")

> > ◦ Reflection: Looking at the different decision trees, which one would you choose for a real-world application?

## Neural network



- Import the MLP neural network classifier algorithm from scikit-learn.

In [ ]:
from sklearn.neural_network import MLPClassifier

- Create an instance of the model and fit it to the training data

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(50, 20)
)
mlp.fit(X_train, y_train)

Generate the predictions

In [ ]:
y_pred = mlp.predict(X_test)

Display a confusion matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)

Compute the accuracy score, precision score and recall score.

In [ ]:
print(f"Accuracy score:  {accuracy_score(y_test,y_pred):.4f}")
print(f"Precision score: {precision_score(y_test,y_pred):.4f}")
print(f"Recall score:    {recall_score(y_test,y_pred):.4f}")

### Try it yourself

- The default `MLPClassifier` architecture consists of one hidden layer with 100 neurons. Experiment with different MLP architectures by changing the `hidden_layer_sizes` parameter.

In [ ]:
layers = (50, 10) #Adjust the architecture by changing the number of hidden layers and neurons per layer.
threshold = 0.5

mlp = MLPClassifier(hidden_layer_sizes=layers)
mlp.fit(X_train, y_train)

y_pred=mlp.predict_proba(X_test)[:,1]>=threshold

print(f"Acc. training:   {mlp.score(X_train,y_train):.4f}")
print(f"Accuracy score:  {mlp.score(X_test,y_test):.4f}")
print(f"Precision score: {precision_score(y_test,y_pred):.4f}")
print(f"Recall score:    {recall_score(y_test,y_pred):.4f}")

ConfusionMatrixDisplay.from_predictions(y_test,y_pred)

- Experiment with different classification thresholds. Compare how changing the threshold affects precision, recall, and the confusion matrix.

In [ ]:
layers = (50, 10) # Adjust the architecture by changing the number of hidden layers and neurons per layer.
threshold = 0.1 # Adjust the classification thresold.

mlp = MLPClassifier(hidden_layer_sizes=layers)
mlp.fit(X_train, y_train)

y_pred=mlp.predict_proba(X_test)[:,1]>=threshold

print(f"Acc. training:   {mlp.score(X_train,y_train):.4f}")
print(f"Accuracy score:  {mlp.score(X_test,y_test):.4f}")
print(f"Precision score: {precision_score(y_test,y_pred):.4f}")
print(f"Recall score:    {recall_score(y_test,y_pred):.4f}")

ConfusionMatrixDisplay.from_predictions(y_test,y_pred)

- Reflection:

  - Does a more complex network perform better?
  - Which architecture would you choose for this problem? Consider both performance and model complexity